Sesión 2 - Spark DataFrames I: Introducción

SparkSession: el punto de entrada unificado

In [5]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-core:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("DataFrames_Dia14")   // nombre visible en la Spark UI
  .master("local[*]")            // modo local, todos los cores disponibles
  .getOrCreate()                 // crea o reutiliza una sesión existente

// Desde SparkSession podemos acceder al SparkContext si lo necesitamos
val sc = spark.sparkContext

println(s"Spark ${spark.version} listo")

import spark.implicits._

Spark 4.1.1 listo


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@2d49cfbb
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@7fecc26b
import spark.implicits._

Crear DataFrames

Hay varias formas de crear un DataFrame según el origen de los datos.

In [4]:
//Desde una colección en memoria
import spark.implicits._
// Forma 1: Seq de tuplas + toDF con nombres de columna
val dfEmpleados = Seq(
  (1, "Ana García",    28, "Ingeniería"),
  (2, "Luis Martínez", 34, "Marketing"),
  (3, "Marta López",   22, "Ingeniería"),
  (4, "Pedro Ruiz",    41, "Dirección")
).toDF("id", "nombre", "edad", "departamento")

dfEmpleados.show()

+---+-------------+----+------------+
| id|       nombre|edad|departamento|
+---+-------------+----+------------+
|  1|   Ana García|  28|  Ingeniería|
|  2|Luis Martínez|  34|   Marketing|
|  3|  Marta López|  22|  Ingeniería|
|  4|   Pedro Ruiz|  41|   Dirección|
+---+-------------+----+------------+



import spark.implicits._
dfEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 2 more fields]

Pasos para crear un .csv y un .json en vscode desde el notebook

Pasos para crear datos sintéticos en csv desde VsCode

✅ 1. Crear CSV sintético en esa ruta:

In [6]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val contenidoCSV =
"""id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""

val pathCSV = s"$ruta/ventas.csv"

Files.write(
  Paths.get(pathCSV),
  contenidoCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")

CSV creado en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
res6_3: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos
contenidoCSV: String = """id_venta,fecha,producto,categoria,cantidad,precio_unitario,ciudad
1,2026-04-01,Portátil,Informática,2,850.50,Madrid
2,2026-04-01,Ratón,Informática,5,18.90,Valencia
3,2026-04-02,Teclado,Informática,3,45.00,Sevilla
4,2026-04-02,Monitor,Informática,1,199.99,Madrid
5,2026-04-03,Silla,Oficina,4,120.00,Barcelona
6,2026-04-03,Mesa,Oficina,2,250.00,Zaragoza
7,2026-04-04,Webcam,Informática,6,39.90,Madrid
8,2026-04-04,Auriculares,Informática,3,59.99,Valencia
"""
pathCSV: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv"
res6_6: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\ventas.csv

In [ ]:
val dfVentas = spark.read
  .option("header", "true") //si tiene o no cabecera
  .option("inferSchema", "true") // si va a inferir o no el esquema automaticamente
  .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv")

dfVentas.show()
dfVentas.printSchema()

+--------+----------+-----------+-----------+--------+---------------+---------+
|id_venta|     fecha|   producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+-----------+-----------+--------+---------------+---------+
|       1|2026-04-01|   Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|      Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02|    Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02|    Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|      Silla|    Oficina|       4|          120.0|Barcelona|
|       6|2026-04-03|       Mesa|    Oficina|       2|          250.0| Zaragoza|
|       7|2026-04-04|     Webcam|Informática|       6|           39.9|   Madrid|
|       8|2026-04-04|Auriculares|Informática|       3|          59.99| Valencia|
+--------+----------+-----------+-----------+--------+---------------+---------+

root
 |-- id_venta: integer

dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

Pasos para crear un JSON simple:

En Spark, el JSON simple suele ser JSON Lines: un objeto JSON por línea.

In [11]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos" // Usa tu propia ruta, sino te dará error
Files.createDirectories(Paths.get(ruta))

val jsonSimple =
"""{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
{"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
{"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
{"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""

val rutaJsonSimple = s"$ruta/empleados_simple.json"

Files.write(
  Paths.get(rutaJsonSimple),
  jsonSimple.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON simple creado en: $rutaJsonSimple")

JSON simple creado en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_simple.json


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
res11_3: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos
jsonSimple: String = """{"id":1,"nombre":"Ana García","edad":28,"departamento":"Ingeniería"}
{"id":2,"nombre":"Luis Martínez","edad":34,"departamento":"Marketing"}
{"id":3,"nombre":"Marta López","edad":22,"departamento":"Ingeniería"}
{"id":4,"nombre":"Pedro Ruiz","edad":41,"departamento":"Dirección"}"""
rutaJsonSimple: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_simple.json"
res11_6: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\empleados_simple.json

In [12]:
val dfJsonSimple = spark.read
  .option("inferSchema", "true")
  .json(rutaJsonSimple)

dfJsonSimple.show(false)
dfJsonSimple.printSchema()

+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonSimple: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

Pasos para crear un JSON multilínea:

Este formato tiene normalmente un array JSON con varios objetos dentro.

In [13]:
val jsonMultilinea =
"""[
  {
    "id": 1,
    "nombre": "Ana García",
    "edad": 28,
    "departamento": "Ingeniería"
  },
  {
    "id": 2,
    "nombre": "Luis Martínez",
    "edad": 34,
    "departamento": "Marketing"
  },
  {
    "id": 3,
    "nombre": "Marta López",
    "edad": 22,
    "departamento": "Ingeniería"
  },
  {
    "id": 4,
    "nombre": "Pedro Ruiz",
    "edad": 41,
    "departamento": "Dirección"
  }
]"""

val rutaJsonMultilinea = s"$ruta/empleados_multilinea.json"

Files.write(
  Paths.get(rutaJsonMultilinea),
  jsonMultilinea.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON multilínea creado en: $rutaJsonMultilinea")

JSON multilínea creado en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json


jsonMultilinea: String = """[
  {
    "id": 1,
    "nombre": "Ana García",
    "edad": 28,
    "departamento": "Ingeniería"
  },
  {
    "id": 2,
    "nombre": "Luis Martínez",
    "edad": 34,
    "departamento": "Marketing"
  },
  {
    "id": 3,
    "nombre": "Marta López",
    "edad": 22,
    "departamento": "Ingeniería"
  },
  {
    "id": 4,
    "nombre": "Pedro Ruiz",
    "edad": 41,
    "departamento": "Dirección"
  }
]"""
rutaJsonMultilinea: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json"
res13_2: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\empleados_multilinea.json

In [15]:
val dfJsonMultilinea = spark.read
  .option("multiline", "true")
  .option("inferSchema", "true")
  .json(rutaJsonMultilinea)

dfJsonMultilinea.show(false)
dfJsonMultilinea.printSchema()

+------------+----+---+-------------+
|departamento|edad|id |nombre       |
+------------+----+---+-------------+
|Ingeniería  |28  |1  |Ana García   |
|Marketing   |34  |2  |Luis Martínez|
|Ingeniería  |22  |3  |Marta López  |
|Dirección   |41  |4  |Pedro Ruiz   |
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfJsonMultilinea: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

Desde un fichero CSV

In [18]:
val dfVentas = spark.read
  .option("header", "true")       // primera fila como nombres de columna
  .option("inferSchema", "true")  // detectar tipos automáticamente
  .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/ventas.csv")

dfVentas.show()         // muestra las primeras 5 filas


+--------+----------+-----------+-----------+--------+---------------+---------+
|id_venta|     fecha|   producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+-----------+-----------+--------+---------------+---------+
|       1|2026-04-01|   Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|      Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02|    Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02|    Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|      Silla|    Oficina|       4|          120.0|Barcelona|
|       6|2026-04-03|       Mesa|    Oficina|       2|          250.0| Zaragoza|
|       7|2026-04-04|     Webcam|Informática|       6|           39.9|   Madrid|
|       8|2026-04-04|Auriculares|Informática|       3|          59.99| Valencia|
+--------+----------+-----------+-----------+--------+---------------+---------+



dfVentas: org.apache.spark.sql.package.DataFrame = [id_venta: int, fecha: date ... 5 more fields]

In [17]:
dfVentas.printSchema()   // muestra la estructura con tipos

root
 |-- id_venta: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- ciudad: string (nullable = true)



4.3 Desde un fichero JSON

In [20]:
val dfClientes = spark.read
  .option("multiline", "true")    // para JSON con objetos multilínea
  .json("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/empleados_multilinea.json")

dfClientes.show()
dfClientes.printSchema()

+------------+----+---+-------------+
|departamento|edad| id|       nombre|
+------------+----+---+-------------+
|  Ingeniería|  28|  1|   Ana García|
|   Marketing|  34|  2|Luis Martínez|
|  Ingeniería|  22|  3|  Marta López|
|   Dirección|  41|  4|   Pedro Ruiz|
+------------+----+---+-------------+

root
 |-- departamento: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)



dfClientes: org.apache.spark.sql.package.DataFrame = [departamento: string, edad: bigint ... 2 more fields]

Schema: la estructura del DataFrame

El schema describe las columnas del DataFrame: sus nombres, tipos de dato y si admiten valores nulos.

Es la información que hace que Spark pueda optimizar las operaciones.

5.1 Inspeccionar el schema con printSchema()

dfEmpleados.printSchema()

// 

//  |-- id: integer (nullable = true)

//  |-- nombre: string (nullable = true)

//  |-- edad: integer (nullable = true)

//  |-- departamento: string (nullable = true)

La salida muestra un árbol con:

Nombre de cada columna

Tipo de dato (integer, string, double, boolean, date, ...)

nullable: si la columna puede contener valores nulos (true = sí puede)

5.2 Schema por inferencia vs. schema manual
Cuando usamos inferSchema = true, Spark lee una muestra del fichero y adivina los tipos. Es cómodo pero tiene dos inconvenientes: es más lento (tiene que leer los datos dos veces) y puede equivocarse con columnas ambiguas (por ejemplo, un código postal "01234" podría inferirse como integer y perder el cero inicial).

La alternativa es definir el schema manualmente con StructType y StructField:

In [23]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

import org.apache.spark.sql.types._

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
Files.createDirectories(Paths.get(ruta))

val contenidoPedidos =
"""id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""

val rutaPedidos = s"$ruta/pedidos.csv"

Files.write(
  Paths.get(rutaPedidos),
  contenidoPedidos.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado correctamente en: $rutaPedidos")

CSV creado correctamente en: C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
import org.apache.spark.sql.types._
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos"
res23_4: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos
contenidoPedidos: String = """id_pedido,id_cliente,fecha,importe,completado
1,101,2026-04-01,250.75,true
2,102,2026-04-01,99.90,false
3,103,2026-04-02,430.00,true
4,101,2026-04-03,120.50,true
5,104,2026-04-04,75.25,false
6,105,2026-04-05,680.40,true
"""
rutaPedidos: String = "C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv"
res23_7: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\Notebooks\datos\pedidos.csv

In [26]:
import org.apache.spark.sql.types._
val schemaPedidos = StructType(List(
  StructField("id_pedido", IntegerType, nullable = false),
  StructField("id_cliente", IntegerType, nullable = false),
  StructField("fecha", StringType, nullable = true),
  StructField("importe", DoubleType, nullable = true),
  StructField("completado", BooleanType, nullable = true)
))

val dfPedidos = spark.read
  .option("header", "true")
  .schema(schemaPedidos)
  .csv("C:/Users/Imp_06 - Mañana/Desktop/Notebooks/datos/pedidos.csv")

dfPedidos.show(false)

dfPedidos.printSchema()


+---------+----------+----------+-------+----------+
|id_pedido|id_cliente|fecha     |importe|completado|
+---------+----------+----------+-------+----------+
|1        |101       |2026-04-01|250.75 |true      |
|2        |102       |2026-04-01|99.9   |false     |
|3        |103       |2026-04-02|430.0  |true      |
|4        |101       |2026-04-03|120.5  |true      |
|5        |104       |2026-04-04|75.25  |false     |
|6        |105       |2026-04-05|680.4  |true      |
+---------+----------+----------+-------+----------+

root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- completado: boolean (nullable = true)



import org.apache.spark.sql.types._
schemaPedidos: StructType = Seq(
  StructField(
    name = "id_pedido",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "id_cliente",
    dataType = IntegerType,
    nullable = false,
    metadata = {}
  ),
  StructField(
    name = "fecha",
    dataType = StringType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "importe",
    dataType = DoubleType,
    nullable = true,
    metadata = {}
  ),
  StructField(
    name = "completado",
    dataType = BooleanType,
    nullable = true,
    metadata = {}
  )
)
dfPedidos: org.apache.spark.sql.package.DataFrame = [id_pedido: int, id_cliente: int ... 3 more fields]

EJERCICIO #1

🗂️ Paso previo: crear los ficheros de datos

Antes de empezar con el notebook, crea los ficheros de datos que usaremos. Abre el Bloc de notas o

cualquier editor de texto como VsCode y guarda los siguientes ficheros en C:\Curso-Scala\datos\ 

 (crea la carpeta si no existe).

In [32]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/empleados_16"
Files.createDirectories(Paths.get(ruta))

val contenidoCSV =
"""id,nombre,edad,departamento,salario,activo
1,Ana García,28,Ingeniería,42000,true
2,Luis Martínez,34,Marketing,38000,true
3,Marta López,22,Ingeniería,35000,true
4,Pedro Ruiz,41,Dirección,75000,true
5,Carmen Díaz,29,Marketing,36500,true
6,Jorge Santos,38,Ingeniería,48000,false
7,Elena Vega,31,RRHH,33000,true
8,Tomás Gil,45,Dirección,82000,true
9,Laura Prieto,26,Ingeniería,39000,true
10,Andrés Mora,33,Marketing,41000,false
"""

val pathCSV = s"$ruta/empleados.csv"

Files.write(
  Paths.get(pathCSV),
  contenidoCSV.getBytes(StandardCharsets.UTF_8)
)

println(s"CSV creado en: $pathCSV")

CSV creado en: C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/empleados_16/empleados.csv


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/empleados_16"
res32_3: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\ProyectoScalaVSC\Practicas_scala\empleados_16
contenidoCSV: String = """id,nombre,edad,departamento,salario,activo
1,Ana García,28,Ingeniería,42000,true
2,Luis Martínez,34,Marketing,38000,true
3,Marta López,22,Ingeniería,35000,true
4,Pedro Ruiz,41,Dirección,75000,true
5,Carmen Díaz,29,Marketing,36500,true
6,Jorge Santos,38,Ingeniería,48000,false
7,Elena Vega,31,RRHH,33000,true
8,Tomás Gil,45,Dirección,82000,true
9,Laura Prieto,26,Ingeniería,39000,true
10,Andrés Mora,33,Marketing,41000,false
"""
pathCSV: String = "C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/empleados_16/empleados.csv"
res32_6: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\ProyectoScalaVSC\Practicas_scala\empleados_16\empleados.csv

In [36]:
val dfVentasEmpleados = spark.read
  .option("header", "true") //si tiene o no cabecera
  .option("inferSchema", "true") // si va a inferir o no el esquema automaticamente
  .csv("C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/empleados_16/empleados.csv")

dfVentas.show()
dfVentas.printSchema()

+--------+----------+-----------+-----------+--------+---------------+---------+
|id_venta|     fecha|   producto|  categoria|cantidad|precio_unitario|   ciudad|
+--------+----------+-----------+-----------+--------+---------------+---------+
|       1|2026-04-01|   Portátil|Informática|       2|          850.5|   Madrid|
|       2|2026-04-01|      Ratón|Informática|       5|           18.9| Valencia|
|       3|2026-04-02|    Teclado|Informática|       3|           45.0|  Sevilla|
|       4|2026-04-02|    Monitor|Informática|       1|         199.99|   Madrid|
|       5|2026-04-03|      Silla|    Oficina|       4|          120.0|Barcelona|
|       6|2026-04-03|       Mesa|    Oficina|       2|          250.0| Zaragoza|
|       7|2026-04-04|     Webcam|Informática|       6|           39.9|   Madrid|
|       8|2026-04-04|Auriculares|Informática|       3|          59.99| Valencia|
+--------+----------+-----------+-----------+--------+---------------+---------+

root
 |-- id_venta: integer

dfVentasEmpleados: org.apache.spark.sql.package.DataFrame = [id: int, nombre: string ... 4 more fields]

AHORA LO MISMO PERO CON .JSON

In [37]:
import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets

val ruta = "C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/Clientes_16" // Usa tu propia ruta, sino te dará error
Files.createDirectories(Paths.get(ruta))

val jsonSimple =
"""{"id": 101, "nombre": "Laptop Pro", "categoria": "Informatica", "precio": 1299.99, "stock": 45},
  {"id": 102, "nombre": "Teclado Inalámbrico", "categoria": "Perifericos", "precio": 59.90, "stock": 120},
  {"id": 103, "nombre": "Monitor 27\"", "categoria": "Monitores", "precio": 349.00, "stock": 30},
  {"id": 104, "nombre": "Ratón Óptico", "categoria": "Perifericos", "precio": 24.95, "stock": 200},
  {"id": 105, "nombre": "Auriculares USB", "categoria": "Audio", "precio": 89.50, "stock": 75},
  {"id": 106, "nombre": "Webcam HD", "categoria": "Perifericos", "precio": 79.00, "stock": 60},
  {"id": 107, "nombre": "Disco SSD 1TB", "categoria": "Almacenamiento", "precio": 119.99, "stock": 90},
  {"id": 108, "nombre": "Hub USB-C", "categoria": "Perifericos", "precio": 44.90, "stock": 150}"""

val rutaJsonSimple = s"$ruta/clientes.json"

Files.write(
  Paths.get(rutaJsonSimple),
  jsonSimple.getBytes(StandardCharsets.UTF_8)
)

println(s"JSON simple creado en: $rutaJsonSimple")

JSON simple creado en: C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/Clientes_16/clientes.json


import java.nio.file.{Files, Paths}
import java.nio.charset.StandardCharsets
ruta: String = "C:/Users/Imp_06 - Mañana/Desktop/ProyectoScalaVSC/Practicas_scala/Clientes_16"
res37_3: java.nio.file.Path = C:\Users\Imp_06 - Mañana\Desktop\ProyectoScalaVSC\Practicas_scala\Clientes_16
jsonSimple: String = """{"id": 101, "nombre": "Laptop Pro", "categoria": "Informatica", "precio": 1299.99, "stock": 45},
  {"id": 102, "nombre": "Teclado Inalámbrico", "categoria": "Perifericos", "precio": 59.90, "stock": 120},
  {"id": 103, "nombre": "Monitor 27\"", "categoria": "Monitores", "precio": 349.00, "stock": 30},
  {"id": 104, "nombre": "Ratón Óptico", "categoria": "Perifericos", "precio": 24.95, "stock": 200},
  {"id": 105, "nombre": "Auriculares USB", "categoria": "Audio", "precio": 89.50, "stock": 75},
  {"id": 106, "nombre": "Webcam HD", "categoria": "Perifericos", "precio": 79.00, "stock": 60},
  {"id": 107, "nombre": "Disco SSD 1TB", "categoria": "Almacenamiento", "precio": 119.99, "stock": 90}

In [40]:
val dfclientes= spark.read
  .option("inferSchema", "true")
  .json(rutaJsonSimple)

dfclientes.show(false)
dfclientes.printSchema()

+--------------+---+-------------------+-------+-----+
|categoria     |id |nombre             |precio |stock|
+--------------+---+-------------------+-------+-----+
|Informatica   |101|Laptop Pro         |1299.99|45   |
|Perifericos   |102|Teclado Inalámbrico|59.9   |120  |
|Monitores     |103|Monitor 27"        |349.0  |30   |
|Perifericos   |104|Ratón Óptico       |24.95  |200  |
|Audio         |105|Auriculares USB    |89.5   |75   |
|Perifericos   |106|Webcam HD          |79.0   |60   |
|Almacenamiento|107|Disco SSD 1TB      |119.99 |90   |
|Perifericos   |108|Hub USB-C          |44.9   |150  |
+--------------+---+-------------------+-------+-----+

root
 |-- categoria: string (nullable = true)
 |-- id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- precio: double (nullable = true)
 |-- stock: long (nullable = true)



dfclientes: org.apache.spark.sql.package.DataFrame = [categoria: string, id: bigint ... 3 more fields]

DataFrame desde colección en memoria

Crea tu primer DataFrame desde datos en el propio notebook y practica los métodos de exploración.

In [41]:
// Crear DataFrame desde una secuencia de tuplas
val dfCursos = Seq(
  (1, "Big Data con Scala", "Avanzado",  140, 4.8),
  (2, "Python para Datos",  "Intermedio", 80, 4.6),
  (3, "SQL Empresarial",    "Básico",     40, 4.9),
  (4, "Machine Learning",   "Avanzado",  120, 4.7),
  (5, "Power BI",           "Básico",     60, 4.5)
).toDF("id", "titulo", "nivel", "horas", "valoracion")

// Ver las filas
println("=== Contenido del DataFrame ===")
dfCursos.show()

// Ver la estructura
println("=== Schema del DataFrame ===")
dfCursos.printSchema()

// Número de filas y columnas
println(s"Filas: ${dfCursos.count()}")
println(s"Columnas: ${dfCursos.columns.length}")
println(s"Nombres de columnas: ${dfCursos.columns.mkString(", ")}")

=== Contenido del DataFrame ===
+---+------------------+----------+-----+----------+
| id|            titulo|     nivel|horas|valoracion|
+---+------------------+----------+-----+----------+
|  1|Big Data con Scala|  Avanzado|  140|       4.8|
|  2| Python para Datos|Intermedio|   80|       4.6|
|  3|   SQL Empresarial|    Básico|   40|       4.9|
|  4|  Machine Learning|  Avanzado|  120|       4.7|
|  5|          Power BI|    Básico|   60|       4.5|
+---+------------------+----------+-----+----------+

=== Schema del DataFrame ===
root
 |-- id: integer (nullable = false)
 |-- titulo: string (nullable = true)
 |-- nivel: string (nullable = true)
 |-- horas: integer (nullable = false)
 |-- valoracion: double (nullable = false)

Filas: 5
Columnas: 5
Nombres de columnas: id, titulo, nivel, horas, valoracion


dfCursos: org.apache.spark.sql.package.DataFrame = [id: int, titulo: string ... 3 more fields]

In [42]:
// Estadísticas descriptivas de columnas numéricas
println("=== Estadísticas descriptivas ===")
dfCursos.describe("horas", "valoracion").show()

=== Estadísticas descriptivas ===
+-------+-----------------+------------------+
|summary|            horas|        valoracion|
+-------+-----------------+------------------+
|  count|                5|                 5|
|   mean|             88.0|               4.7|
| stddev|41.47288270665544|0.1581138830084192|
|    min|               40|               4.5|
|    max|              140|               4.9|
+-------+-----------------+------------------+



In [ ]:
val dfEmpleados = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("C:/Curso-Scala/datos/empleados.csv")

println("=== Primeras filas ===")
dfEmpleados.show()

println("=== Schema inferido ===")
dfEmpleados.printSchema()